# Advanced Problems with Solutions — Python Properties
## Tutorial-Style Practice Notebook

This notebook is intentionally written in the same **walk-through style** as the provided
lesson: we will start with a concrete class, inspect its behavior, make a change, inspect
again, and only then generalize what happened.

The subject is still **instance properties**.

The property itself will be defined on the class, while the values managed by the property
will normally be different for every instance.

We will repeatedly use:

- dotted notation: `obj.x`
- `getattr(obj, 'x')`
- `setattr(obj, 'x', value)`
- `delattr(obj, 'x')`
- instance dictionaries: `obj.__dict__`
- class dictionaries: `SomeClass.__dict__`
- the `property` callable

### How these exercises are organized

Each problem is broken into small logical steps.

We will often do this:

1. Build the simplest version first.
2. Observe a weakness.
3. Add accessor methods.
4. Turn those accessors into a property.
5. Inspect the instance and class dictionaries.
6. Try edge cases.
7. Extend the design.

This means some classes will be redefined several times on purpose.
That is part of the tutorial.

### A small helper for our experiments

Many of our examples deliberately trigger exceptions.

Rather than allowing those exceptions to stop the notebook, we will use a helper that prints
the exception type and message.

In [1]:
def show_exception(func, *args, **kwargs):
    try:
        return func(*args, **kwargs)
    except Exception as ex:
        print(f"{type(ex).__name__}: {ex}")

# Problem 1 — Evolving a Public Attribute Without Changing the Interface

Suppose we are writing a `Product` class.

We start with a plain instance attribute called `price`.
At this point we do not need any special logic.

In [2]:
class Product:
    def __init__(self, price):
        self.price = price

In [3]:
p = Product(19.95)
p.price

19.95

Nothing special is happening yet.

`price` is simply stored in the instance dictionary.

In [4]:
p.__dict__

{'price': 19.95}

Now imagine that our class is already used by other code.

Users are doing this:

```python
p.price
p.price = 25
```

Later, we decide that negative prices should not be allowed.

One possibility is to replace the public attribute with explicit methods such as
`get_price()` and `set_price()`.

Let us see that version first.

In [5]:
class Product:
    def __init__(self, price):
        self.set_price(price)

    def get_price(self):
        return self._price

    def set_price(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("price must be a number")
        if value < 0:
            raise ValueError("price cannot be negative")
        self._price = float(value)

In [6]:
p = Product(19.95)
p.get_price()

19.95

This works, but we changed the interface.

Existing code that expects `p.price` would now have to use `p.get_price()`.

Instead, we can keep the accessor methods and create a property named `price`.

In [7]:
class Product:
    def __init__(self, price):
        self.price = price

    def get_price(self):
        return self._price

    def set_price(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("price must be a number")
        if value < 0:
            raise ValueError("price cannot be negative")
        self._price = float(value)

    price = property(fget=get_price, fset=set_price)

In [8]:
p = Product(19.95)

print(p.price)
p.price = 25
print(p.price)

19.95
25.0


So the public interface is once again the simple dotted interface.

But internally, reading `p.price` calls `get_price`, and assigning to `p.price`
calls `set_price`.

In [9]:
show_exception(setattr, p, "price", -1)
show_exception(setattr, p, "price", "free")

ValueError: price cannot be negative
TypeError: price must be a number


Let us compare the instance dictionary with the class dictionary.

In [10]:
p.__dict__

{'_price': 25.0}

In [11]:
Product.__dict__["price"]

Notice the split:

- `_price` is instance-specific storage.
- `price` is a property object stored on the class.

This is exactly what allows every `Product` instance to have a different price while sharing
the same access-management logic.

In [12]:
p1 = Product(10)
p2 = Product(99)

print(p1.__dict__)
print(p2.__dict__)
print(p1.price, p2.price)

{'_price': 10.0}
{'_price': 99.0}
10.0 99.0


### Solution takeaway

We were able to start with a bare public attribute and later add validation without changing
the dotted interface used by callers.

This is one of the reasons Python code does not need to begin with Java-style getter and
setter methods for every attribute.

# Problem 2 — What Happens if the Instance Dictionary Contains the Same Name?

This problem is more subtle.

We will create a managed `username` property backed by `_username`.

Then we will deliberately place another key called `username` directly inside the instance
dictionary.

Before running the later cells, predict what dotted access will return.

In [13]:
class Account:
    def __init__(self, username):
        self.username = username

    def get_username(self):
        print("getter called")
        return self._username

    def set_username(self, value):
        print("setter called")
        if not isinstance(value, str):
            raise TypeError("username must be a string")

        value = value.strip()
        if not value:
            raise ValueError("username cannot be empty")

        self._username = value

    username = property(get_username, set_username)

In [14]:
a = Account("alex")
a.__dict__

setter called


{'_username': 'alex'}

So far there is no surprise.

Now we will bypass the normal interface and manually add a key named `username`.

In [15]:
a.__dict__["username"] = "INJECTED"

a.__dict__

{'_username': 'alex', 'username': 'INJECTED'}

There are now two relevant values in the instance dictionary:

- `_username`
- `username`

Which one will `a.username` use?

In [16]:
a.username

getter called


'alex'

The getter was still called.

The manually injected `username` key did **not** replace the behavior of the property.

Let us also assign through dotted notation.

In [17]:
a.username = "raymond"

a.__dict__

setter called


{'_username': 'raymond', 'username': 'INJECTED'}

The setter updated `_username`.

The unrelated injected `username` key is still sitting in the dictionary, but normal dotted
access is still controlled by the property.

In [18]:
print("dotted:", a.username)
print("raw injected key:", a.__dict__["username"])

getter called
dotted: raymond
raw injected key: INJECTED


### Why?

A property with a setter participates in Python's descriptor machinery as a **data descriptor**.

Data descriptors take precedence over an instance dictionary entry of the same public name.

We do not need the full descriptor protocol yet to use properties correctly, but this behavior
explains why simply inserting `"username"` into `a.__dict__` does not fool `a.username`.

### A useful rule

If you want the property to manage a value, use a **different backing name**, such as
`_username`.

Do not try to store the managed value under the exact same public name.

# Problem 3 — Two Properties That Must Agree With Each Other

Validating one property is straightforward.

But what if the validity of one property depends on another?

We will model a date range using integers for simplicity:

- `start`
- `end`

We require:

```text
start <= end
```

The tricky part is initialization because one value may not exist yet when the other setter
runs.

### Step 1 — A naive implementation

Let us write setters that immediately compare against the other property.

In [19]:
class NaiveRange:
    def __init__(self, start, end):
        self.start = start
        self.end = end

    def get_start(self):
        return self._start

    def set_start(self, value):
        if value > self.end:
            raise ValueError("start cannot be greater than end")
        self._start = value

    def get_end(self):
        return self._end

    def set_end(self, value):
        if value < self.start:
            raise ValueError("end cannot be less than start")
        self._end = value

    start = property(get_start, set_start)
    end = property(get_end, set_end)

What happens when we construct the first instance?

The constructor executes `self.start = start` before `_end` exists.

In [20]:
show_exception(NaiveRange, 10, 20)

AttributeError: 'NaiveRange' object has no attribute '_end'


So the validation rule is reasonable, but our initialization sequence is not.

We need each setter to tolerate the period where the other backing attribute has not yet been
created.

### Step 2 — Check whether the other value exists

We can inspect `self.__dict__` before making the cross-field comparison.

In [21]:
class Range:
    def __init__(self, start, end):
        self.start = start
        self.end = end

    def get_start(self):
        return self._start

    def set_start(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("start must be an integer")

        if "_end" in self.__dict__ and value > self._end:
            raise ValueError("start cannot be greater than end")

        self._start = value

    def get_end(self):
        return self._end

    def set_end(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("end must be an integer")

        if "_start" in self.__dict__ and value < self._start:
            raise ValueError("end cannot be less than start")

        self._end = value

    start = property(get_start, set_start)
    end = property(get_end, set_end)

In [22]:
r = Range(10, 20)

print(r.start, r.end)
print(r.__dict__)

10 20
{'_start': 10, '_end': 20}


Now initialization works.

Let us try valid changes.

In [23]:
r.start = 12
r.end = 30

print(r.start, r.end)

12 30


And now invalid changes.

In [24]:
show_exception(setattr, r, "start", 100)
show_exception(setattr, r, "end", 5)

print(r.start, r.end)

ValueError: start cannot be greater than end
ValueError: end cannot be less than start
12 30


Notice something important: because we validate **before** writing the backing attribute,
the previous valid state remains unchanged when an assignment fails.

That is usually what we want.

### Step 3 — Construction with an invalid pair

Even though the first assignment may temporarily exist by itself, the second assignment still
enforces the final invariant.

In [25]:
show_exception(Range, 50, 10)

ValueError: end cannot be less than start


### Solution takeaway

A property can protect invariants that involve several pieces of instance state.

But when properties depend on each other, initialization order becomes part of the design.

# Problem 4 — Two Public Properties, One Stored Value

Suppose we want a temperature object that users can read and write in either Celsius or
Fahrenheit.

A tempting design is to store both:

```python
self._celsius
self._fahrenheit
```

But then the two values can become inconsistent.

We will instead keep **one canonical value** and derive the other.

### Step 1 — Store only Celsius

First we implement `celsius`.

In [26]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    def get_celsius(self):
        return self._celsius

    def set_celsius(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("celsius must be numeric")

        value = float(value)

        if value < -273.15:
            raise ValueError("temperature cannot be below absolute zero")

        self._celsius = value

    celsius = property(get_celsius, set_celsius)

In [27]:
t = Temperature(20)
print(t.celsius)
print(t.__dict__)

20.0
{'_celsius': 20.0}


### Step 2 — Add a computed Fahrenheit getter

Fahrenheit does not need separate storage.

In [28]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    def get_celsius(self):
        return self._celsius

    def set_celsius(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("celsius must be numeric")

        value = float(value)

        if value < -273.15:
            raise ValueError("temperature cannot be below absolute zero")

        self._celsius = value

    def get_fahrenheit(self):
        return self._celsius * 9 / 5 + 32

    celsius = property(get_celsius, set_celsius)
    fahrenheit = property(get_fahrenheit)

In [29]:
t = Temperature(0)

print("C:", t.celsius)
print("F:", t.fahrenheit)
print(t.__dict__)

C: 0.0
F: 32.0
{'_celsius': 0.0}


`fahrenheit` is currently read-only because we supplied only a getter.

Let us verify that.

In [30]:
show_exception(setattr, t, "fahrenheit", 212)

AttributeError: property 'fahrenheit' of 'Temperature' object has no setter


### Step 3 — Make Fahrenheit writable too

The Fahrenheit setter can convert the input to Celsius and then assign through the Celsius
property.

This is useful because it reuses the validation already implemented by `set_celsius`.

In [31]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    def get_celsius(self):
        return self._celsius

    def set_celsius(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("celsius must be numeric")

        value = float(value)

        if value < -273.15:
            raise ValueError("temperature cannot be below absolute zero")

        self._celsius = value

    def get_fahrenheit(self):
        return self._celsius * 9 / 5 + 32

    def set_fahrenheit(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("fahrenheit must be numeric")

        celsius = (float(value) - 32) * 5 / 9
        self.celsius = celsius

    celsius = property(get_celsius, set_celsius)
    fahrenheit = property(get_fahrenheit, set_fahrenheit)

In [32]:
t = Temperature(0)

t.fahrenheit = 212

print("C:", t.celsius)
print("F:", t.fahrenheit)
print(t.__dict__)

C: 100.0
F: 212.0
{'_celsius': 100.0}


We still have only one stored temperature value.

The two public interfaces cannot drift apart because Fahrenheit is always derived from the same
canonical Celsius state.

In [33]:
show_exception(setattr, t, "fahrenheit", -500)
print(t.celsius, t.fahrenheit)

ValueError: temperature cannot be below absolute zero
100.0 212.0


### Solution takeaway

Properties are useful for presenting multiple views of one internal value.

When two values represent the same underlying fact, one canonical storage representation is
usually safer than storing both.

# Problem 5 — A Property With a Meaningful Deleter

Deleting a property does not delete the property object from the class.

A deleter normally changes or removes **instance state**.

We will model an API credential that can be assigned, read, and revoked.

### Step 1 — Getter and setter

First we create the credential without deletion support.

In [34]:
class Credential:
    def __init__(self, token):
        self.token = token

    def get_token(self):
        return self._token

    def set_token(self, value):
        if not isinstance(value, str):
            raise TypeError("token must be a string")

        value = value.strip()

        if not value:
            raise ValueError("token cannot be empty")

        self._token = value

    token = property(get_token, set_token)

In [35]:
c = Credential(" secret-123 ")

print(c.token)
print(c.__dict__)

secret-123
{'_token': 'secret-123'}


If we try `del c.token`, Python has no deleter to call.

In [36]:
show_exception(delattr, c, "token")

AttributeError: property 'token' of 'Credential' object has no deleter


### Step 2 — Add a deleter

The deleter will remove `_token`.

In [37]:
class Credential:
    def __init__(self, token):
        self.token = token

    def get_token(self):
        if "_token" not in self.__dict__:
            raise AttributeError("token has been revoked")
        return self._token

    def set_token(self, value):
        if not isinstance(value, str):
            raise TypeError("token must be a string")

        value = value.strip()

        if not value:
            raise ValueError("token cannot be empty")

        self._token = value

    def del_token(self):
        if "_token" not in self.__dict__:
            raise AttributeError("token is already revoked")
        del self._token

    token = property(get_token, set_token, del_token)

In [38]:
c = Credential("secret-123")

print(c.__dict__)
del c.token
print(c.__dict__)

{'_token': 'secret-123'}
{}


The `_token` value is gone.

But is the property itself gone?

In [39]:
print("token in instance dictionary:", "token" in c.__dict__)
print("token in class dictionary:", "token" in Credential.__dict__)
print(Credential.__dict__["token"])

token in instance dictionary: False
token in class dictionary: True


The property still exists on the class.

Reading it now calls the getter, and the getter discovers that the backing value is absent.

In [40]:
show_exception(getattr, c, "token")

AttributeError: token has been revoked


Because the setter still exists, we can assign a new token later.

In [41]:
c.token = "replacement-token"

print(c.token)
print(c.__dict__)

replacement-token
{'_token': 'replacement-token'}


### Step 3 — The built-in functions behave the same way

We can repeat the operations with `getattr`, `setattr`, and `delattr`.

In [42]:
print(getattr(c, "token"))

setattr(c, "token", "via-setattr")
print(getattr(c, "token"))

delattr(c, "token")
print(c.__dict__)

replacement-token
via-setattr
{}


### Solution takeaway

The class owns the property object.

The instance owns the backing state.

Deleting the managed value does not remove the property definition.

# Problem 6 — A Cached Computed Value That Must Be Invalidated

A property getter can compute a value instead of simply returning a stored attribute.

But if the computation is expensive, we might want to cache it.

That creates another problem: when the inputs change, the cached value can become stale.

We will build a `Report` object whose `normalized_text` property is cached.

### Step 1 — A normal computed getter

We begin without caching.

In [43]:
class Report:
    def __init__(self, text):
        self.text = text

    def get_text(self):
        return self._text

    def set_text(self, value):
        if not isinstance(value, str):
            raise TypeError("text must be a string")
        self._text = value

    def get_normalized_text(self):
        print("computing normalized text")
        return " ".join(self._text.lower().split())

    text = property(get_text, set_text)
    normalized_text = property(get_normalized_text)

In [44]:
report = Report("  Python   Properties  ")

print(report.normalized_text)
print(report.normalized_text)

computing normalized text
python properties
computing normalized text
python properties


The computation runs every time.

Now we will cache the result under `_normalized_text_cache`.

In [45]:
class Report:
    def __init__(self, text):
        self.text = text

    def get_text(self):
        return self._text

    def set_text(self, value):
        if not isinstance(value, str):
            raise TypeError("text must be a string")

        self._text = value

    def get_normalized_text(self):
        if "_normalized_text_cache" not in self.__dict__:
            print("computing normalized text")
            self._normalized_text_cache = " ".join(self._text.lower().split())

        return self._normalized_text_cache

    text = property(get_text, set_text)
    normalized_text = property(get_normalized_text)

In [46]:
report = Report("  Python   Properties  ")

print(report.normalized_text)
print(report.normalized_text)
print(report.__dict__)

computing normalized text
python properties
python properties
{'_text': '  Python   Properties  ', '_normalized_text_cache': 'python properties'}


The second read used the cached value.

But now watch what happens when we change `text`.

In [47]:
report.text = "  Descriptors   and   Properties  "

print("text:", report.text)
print("normalized:", report.normalized_text)

text:   Descriptors   and   Properties  
normalized: python properties


That is a bug.

The cached result still belongs to the old text.

The `text` setter must invalidate the dependent cache.

### Step 2 — Invalidate the cache in the setter

In [48]:
class Report:
    def __init__(self, text):
        self.text = text

    def get_text(self):
        return self._text

    def set_text(self, value):
        if not isinstance(value, str):
            raise TypeError("text must be a string")

        self._text = value
        self.__dict__.pop("_normalized_text_cache", None)

    def get_normalized_text(self):
        if "_normalized_text_cache" not in self.__dict__:
            print("computing normalized text")
            self._normalized_text_cache = " ".join(self._text.lower().split())

        return self._normalized_text_cache

    text = property(get_text, set_text)
    normalized_text = property(get_normalized_text)

In [49]:
report = Report("  Python   Properties  ")

print(report.normalized_text)
print(report.normalized_text)

report.text = "  Descriptors   and   Properties  "

print(report.normalized_text)
print(report.normalized_text)

computing normalized text
python properties
python properties
computing normalized text
descriptors and properties
descriptors and properties


### Step 3 — Inspect the cache directly

The cache is an implementation detail in the instance dictionary.

In [50]:
report.__dict__

{'_text': '  Descriptors   and   Properties  ',
 '_normalized_text_cache': 'descriptors and properties'}

### Design note

A property getter that performs expensive work can be surprising to callers because property
syntax looks like ordinary attribute access.

Caching can help performance, but then invalidation becomes part of the correctness story.

For very expensive or side-effect-heavy operations, a regular method may communicate intent
more clearly.

# Problem 7 — Changing One Accessor in a Subclass

A property object contains separate references to:

- a getter
- a setter
- an optional deleter

A subclass may want to change only one of these behaviors.

We will first see the easy-to-miss mistake.

### Step 1 — Base class

`Score` exposes a validated numeric property.

In [51]:
class Score:
    def __init__(self, value):
        self.value = value

    def get_value(self):
        return self._value

    def set_value(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("score must be an integer")

        if not 0 <= value <= 100:
            raise ValueError("score must be between 0 and 100")

        self._value = value

    value = property(get_value, set_value)

In [52]:
s = Score(85)
print(s.value)

s.value = 90
print(s.value)

85
90


### Step 2 — A subclass creates a new read-only property by accident

Suppose the subclass wants to display the score as `"90/100"`.

If we create a brand-new property with only a getter, the inherited setter is not automatically
copied into that new property object.

In [53]:
class DisplayScoreBroken(Score):
    def __init__(self, value):
        # Store directly only so we can demonstrate the subclass property mistake.
        # If we inherited Score.__init__, its `self.value = value` assignment would
        # immediately fail because this subclass replaced `value` with a read-only property.
        self._value = value

    def get_value(self):
        return f"{self._value}/100"

    value = property(get_value)


In [54]:
broken = DisplayScoreBroken(80)
print(broken.value)

show_exception(setattr, broken, "value", 95)

80/100
AttributeError: property 'value' of 'DisplayScoreBroken' object has no setter


The getter works, but the subclass property has no setter.

In [55]:
print("Base setter:", Score.value.fset)
print("Broken subclass setter:", DisplayScoreBroken.value.fset)

Base setter: <function Score.set_value at 0x0000024F7F2B31A0>
Broken subclass setter: None


### Step 3 — Build a new property while preserving the base setter

Because a property exposes `fget`, `fset`, and `fdel`, we can explicitly reuse the setter.

In [56]:
class DisplayScore(Score):
    def get_display_value(self):
        return f"{self._value}/100"

    value = property(
        fget=get_display_value,
        fset=Score.value.fset,
        fdel=Score.value.fdel,
        doc="Score formatted out of 100."
    )

In [57]:
ds = DisplayScore(80)

print(ds.value)

ds.value = 95
print(ds.value)

80/100
95/100


The subclass getter changed, while assignment still uses the validation logic from the base
class.

In [58]:
show_exception(setattr, ds, "value", 500)

print(DisplayScore.value.fget)
print(DisplayScore.value.fset)
print(DisplayScore.value.__doc__)

ValueError: score must be between 0 and 100
<function DisplayScore.get_display_value at 0x0000024F7F2B37E0>
<function Score.set_value at 0x0000024F7F2B31A0>
Score formatted out of 100.


### Solution takeaway

A property is one object containing several accessor references.

Replacing the entire property in a subclass can unintentionally remove inherited accessor
behavior. If you want to change only one accessor, preserve the others explicitly.

# Problem 8 — Generating Similar Properties

Suppose a class has several positive numeric dimensions:

- `length`
- `width`
- `height`

Writing three almost-identical getter/setter pairs is repetitive.

We can write a function that **creates property objects**.

### Step 1 — Look at the repetition first

A normal explicit version might look like this for one field.

In [59]:
class OneDimension:
    def get_length(self):
        return self._length

    def set_length(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("length must be numeric")

        value = float(value)

        if value <= 0:
            raise ValueError("length must be greater than zero")

        self._length = value

    length = property(get_length, set_length)

We could repeat that pattern three times.

Instead, we can capture the backing attribute name inside a function.

In [60]:
def positive_number_property(storage_name):
    public_name = storage_name.lstrip("_")

    def getter(instance):
        return getattr(instance, storage_name)

    def setter(instance, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f"{public_name} must be numeric")

        value = float(value)

        if value <= 0:
            raise ValueError(f"{public_name} must be greater than zero")

        setattr(instance, storage_name, value)

    return property(
        fget=getter,
        fset=setter,
        doc=f"Positive numeric {public_name}."
    )

The function does not return a number.

It returns a **property object**.

In [61]:
demo_property = positive_number_property("_length")

print(demo_property)
print(type(demo_property))
print(demo_property.fget)
print(demo_property.fset)

<class 'property'>
<function positive_number_property.<locals>.getter at 0x0000024F7F2B3420>
<function positive_number_property.<locals>.setter at 0x0000024F7F2B3560>


### Step 2 — Use the generated properties in a class

In [62]:
class Box:
    length = positive_number_property("_length")
    width = positive_number_property("_width")
    height = positive_number_property("_height")

    def __init__(self, length, width, height):
        self.length = length
        self.width = width
        self.height = height

    def get_volume(self):
        return self.length * self.width * self.height

    volume = property(get_volume)

In [63]:
box = Box(2, 3, 4)

print(box.length, box.width, box.height)
print(box.volume)
print(box.__dict__)

2.0 3.0 4.0
24.0
{'_length': 2.0, '_width': 3.0, '_height': 4.0}


Let us test invalid values.

In [64]:
show_exception(setattr, box, "length", 0)
show_exception(setattr, box, "width", "3")
show_exception(setattr, box, "height", True)

ValueError: length must be greater than zero
TypeError: width must be numeric
TypeError: height must be numeric


### Step 3 — Inspect the generated class properties

In [65]:
print(Box.__dict__["length"])
print(Box.__dict__["width"])
print(Box.__dict__["height"])
print(Box.__dict__["volume"])

Each class attribute is a separate property object.

The generated getter and setter functions remember which backing attribute name they should
use.

### Design note

Property factories can reduce repeated boilerplate, but they also make the class less explicit.

They are most useful when the repeated rule is truly uniform.

If each field starts accumulating different business rules, separate named accessor methods may
be clearer.

# Problem 9 — When a Setter Is the Wrong Abstraction

Properties are powerful, but not every state change should be a property assignment.

Consider a bank account.

We could make `balance` writable, but then callers could do this:

```python
account.balance = -1_000_000
```

We could add validation, but a withdrawal is more than "replace one value with another".

It is a domain operation.

### Step 1 — Read-only balance

We expose balance through a getter only.

In [66]:
class BankAccount:
    def __init__(self, opening_balance=0):
        if isinstance(opening_balance, bool) or not isinstance(opening_balance, (int, float)):
            raise TypeError("opening balance must be numeric")

        if opening_balance < 0:
            raise ValueError("opening balance cannot be negative")

        self._balance = float(opening_balance)

    def get_balance(self):
        return self._balance

    balance = property(get_balance)

In [67]:
account = BankAccount(100)

print(account.balance)
show_exception(setattr, account, "balance", 1000)

100.0
AttributeError: property 'balance' of 'BankAccount' object has no setter


### Step 2 — Add explicit operations

Deposits and withdrawals are verbs, so methods communicate their meaning better.

In [68]:
class BankAccount:
    def __init__(self, opening_balance=0):
        if isinstance(opening_balance, bool) or not isinstance(opening_balance, (int, float)):
            raise TypeError("opening balance must be numeric")

        if opening_balance < 0:
            raise ValueError("opening balance cannot be negative")

        self._balance = float(opening_balance)

    def get_balance(self):
        return self._balance

    def deposit(self, amount):
        if isinstance(amount, bool) or not isinstance(amount, (int, float)):
            raise TypeError("amount must be numeric")

        amount = float(amount)

        if amount <= 0:
            raise ValueError("deposit must be positive")

        self._balance += amount

    def withdraw(self, amount):
        if isinstance(amount, bool) or not isinstance(amount, (int, float)):
            raise TypeError("amount must be numeric")

        amount = float(amount)

        if amount <= 0:
            raise ValueError("withdrawal must be positive")

        if amount > self._balance:
            raise ValueError("insufficient funds")

        self._balance -= amount

    balance = property(get_balance)

In [69]:
account = BankAccount(100)

account.deposit(50)
print(account.balance)

account.withdraw(40)
print(account.balance)

150.0
110.0


In [70]:
show_exception(account.withdraw, 500)
print(account.balance)

ValueError: insufficient funds
110.0


### Why this design is better

A withdrawal may later need:

- audit logging
- authorization
- transaction identifiers
- fees
- notifications
- limits
- rollback behavior

Those responsibilities fit a method much better than an apparently simple assignment.

So an advanced property design skill is knowing when **not** to create a setter.

# Problem 10 — Capstone Tutorial: A Managed `Subscription`

We will now combine several ideas in one class.

A subscription has:

- `plan`: one of `"basic"`, `"pro"`, `"enterprise"`
- `seats`: a positive integer
- `monthly_price`: computed from `plan` and `seats`
- `active`: readable state
- `cancel()`: an operation
- `reactivate()`: an operation

Additional rule:

- `"basic"` allows at most 5 seats
- `"pro"` allows at most 100 seats
- `"enterprise"` has no fixed maximum in this exercise

We will build the class gradually.

## Capstone Step 1 — Manage the plan

We will normalize plan names to lowercase and reject unsupported plans.

In [71]:
class Subscription:
    ALLOWED_PLANS = {"basic", "pro", "enterprise"}

    def __init__(self, plan):
        self.plan = plan

    def get_plan(self):
        return self._plan

    def set_plan(self, value):
        if not isinstance(value, str):
            raise TypeError("plan must be a string")

        value = value.strip().lower()

        if value not in self.ALLOWED_PLANS:
            raise ValueError(f"unsupported plan: {value!r}")

        self._plan = value

    plan = property(get_plan, set_plan)

In [72]:
sub = Subscription(" PRO ")

print(sub.plan)
print(sub.__dict__)

pro
{'_plan': 'pro'}


## Capstone Step 2 — Add seats

Seats must be a positive integer.

At first we will validate only the type and positivity.

In [73]:
class Subscription:
    ALLOWED_PLANS = {"basic", "pro", "enterprise"}

    def __init__(self, plan, seats):
        self.plan = plan
        self.seats = seats

    def get_plan(self):
        return self._plan

    def set_plan(self, value):
        if not isinstance(value, str):
            raise TypeError("plan must be a string")

        value = value.strip().lower()

        if value not in self.ALLOWED_PLANS:
            raise ValueError(f"unsupported plan: {value!r}")

        self._plan = value

    def get_seats(self):
        return self._seats

    def set_seats(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("seats must be an integer")

        if value <= 0:
            raise ValueError("seats must be positive")

        self._seats = value

    plan = property(get_plan, set_plan)
    seats = property(get_seats, set_seats)

In [74]:
sub = Subscription("pro", 10)

print(sub.plan, sub.seats)
print(sub.__dict__)

pro 10
{'_plan': 'pro', '_seats': 10}


## Capstone Step 3 — Enforce plan-specific seat limits

Now `seats` depends on `plan`.

Since `plan` is assigned first in the initializer, the seats setter can safely use
`self.plan`.

In [75]:
class Subscription:
    ALLOWED_PLANS = {"basic", "pro", "enterprise"}
    MAX_SEATS = {
        "basic": 5,
        "pro": 100,
        "enterprise": None,
    }

    def __init__(self, plan, seats):
        self.plan = plan
        self.seats = seats

    def get_plan(self):
        return self._plan

    def set_plan(self, value):
        if not isinstance(value, str):
            raise TypeError("plan must be a string")

        value = value.strip().lower()

        if value not in self.ALLOWED_PLANS:
            raise ValueError(f"unsupported plan: {value!r}")

        self._plan = value

    def get_seats(self):
        return self._seats

    def set_seats(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("seats must be an integer")

        if value <= 0:
            raise ValueError("seats must be positive")

        max_seats = self.MAX_SEATS[self.plan]

        if max_seats is not None and value > max_seats:
            raise ValueError(
                f"{self.plan} plan allows at most {max_seats} seats"
            )

        self._seats = value

    plan = property(get_plan, set_plan)
    seats = property(get_seats, set_seats)

In [76]:
print(Subscription("basic", 5).__dict__)
show_exception(Subscription, "basic", 6)

print(Subscription("pro", 100).__dict__)
show_exception(Subscription, "pro", 101)

print(Subscription("enterprise", 1000).__dict__)

{'_plan': 'basic', '_seats': 5}
ValueError: basic plan allows at most 5 seats
{'_plan': 'pro', '_seats': 100}
ValueError: pro plan allows at most 100 seats
{'_plan': 'enterprise', '_seats': 1000}


There is still a subtle bug.

What if we already have a Pro subscription with 50 seats and then switch the plan to Basic?

The current `plan` setter does not check whether the **existing seat count** is valid for the
new plan.

In [77]:
sub = Subscription("pro", 50)

sub.plan = "basic"

print(sub.plan, sub.seats)

basic 50


We have created an invalid state.

So validation belongs on **both paths** that can affect the invariant:

- changing seats
- changing plan

## Capstone Step 4 — Protect the invariant from either direction

In [78]:
class Subscription:
    ALLOWED_PLANS = {"basic", "pro", "enterprise"}
    MAX_SEATS = {
        "basic": 5,
        "pro": 100,
        "enterprise": None,
    }

    def __init__(self, plan, seats):
        self.plan = plan
        self.seats = seats
        self._active = True

    def get_plan(self):
        return self._plan

    def set_plan(self, value):
        if not isinstance(value, str):
            raise TypeError("plan must be a string")

        value = value.strip().lower()

        if value not in self.ALLOWED_PLANS:
            raise ValueError(f"unsupported plan: {value!r}")

        if "_seats" in self.__dict__:
            max_seats = self.MAX_SEATS[value]

            if max_seats is not None and self._seats > max_seats:
                raise ValueError(
                    f"cannot switch to {value}; "
                    f"{self._seats} seats exceeds its limit of {max_seats}"
                )

        self._plan = value

    def get_seats(self):
        return self._seats

    def set_seats(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("seats must be an integer")

        if value <= 0:
            raise ValueError("seats must be positive")

        max_seats = self.MAX_SEATS[self.plan]

        if max_seats is not None and value > max_seats:
            raise ValueError(
                f"{self.plan} plan allows at most {max_seats} seats"
            )

        self._seats = value

    def get_active(self):
        return self._active

    plan = property(get_plan, set_plan)
    seats = property(get_seats, set_seats)
    active = property(get_active)

In [79]:
sub = Subscription("pro", 50)

show_exception(setattr, sub, "plan", "basic")

print(sub.plan, sub.seats)

ValueError: cannot switch to basic; 50 seats exceeds its limit of 5
pro 50


The failed assignment left the previous valid state intact.

## Capstone Step 5 — Add a computed monthly price

We will use simple pricing for the exercise:

- Basic: 10 per seat
- Pro: 25 per seat
- Enterprise: 60 per seat

There is no reason to store monthly price separately because it can be computed from the
current plan and seat count.

In [80]:
class Subscription:
    ALLOWED_PLANS = {"basic", "pro", "enterprise"}
    MAX_SEATS = {
        "basic": 5,
        "pro": 100,
        "enterprise": None,
    }
    PRICE_PER_SEAT = {
        "basic": 10.0,
        "pro": 25.0,
        "enterprise": 60.0,
    }

    def __init__(self, plan, seats):
        self.plan = plan
        self.seats = seats
        self._active = True

    def get_plan(self):
        return self._plan

    def set_plan(self, value):
        if not isinstance(value, str):
            raise TypeError("plan must be a string")

        value = value.strip().lower()

        if value not in self.ALLOWED_PLANS:
            raise ValueError(f"unsupported plan: {value!r}")

        if "_seats" in self.__dict__:
            max_seats = self.MAX_SEATS[value]

            if max_seats is not None and self._seats > max_seats:
                raise ValueError(
                    f"cannot switch to {value}; "
                    f"{self._seats} seats exceeds its limit of {max_seats}"
                )

        self._plan = value

    def get_seats(self):
        return self._seats

    def set_seats(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("seats must be an integer")

        if value <= 0:
            raise ValueError("seats must be positive")

        max_seats = self.MAX_SEATS[self.plan]

        if max_seats is not None and value > max_seats:
            raise ValueError(
                f"{self.plan} plan allows at most {max_seats} seats"
            )

        self._seats = value

    def get_monthly_price(self):
        return self.PRICE_PER_SEAT[self.plan] * self.seats

    def get_active(self):
        return self._active

    plan = property(get_plan, set_plan)
    seats = property(get_seats, set_seats)
    monthly_price = property(get_monthly_price)
    active = property(get_active)

In [81]:
sub = Subscription("pro", 12)

print(sub.plan)
print(sub.seats)
print(sub.monthly_price)
print(sub.__dict__)

pro
12
300.0
{'_plan': 'pro', '_seats': 12, '_active': True}


Notice that there is no `_monthly_price` in the instance dictionary.

It is derived each time from current state.

In [82]:
sub.seats = 20

print(sub.monthly_price)
print(sub.__dict__)

500.0
{'_plan': 'pro', '_seats': 20, '_active': True}


## Capstone Step 6 — Add domain operations instead of an `active` setter

We do not want callers to write arbitrary values such as:

```python
sub.active = "maybe"
```

Activation and cancellation are domain operations, so we will use methods.

In [83]:
class Subscription:
    ALLOWED_PLANS = {"basic", "pro", "enterprise"}
    MAX_SEATS = {
        "basic": 5,
        "pro": 100,
        "enterprise": None,
    }
    PRICE_PER_SEAT = {
        "basic": 10.0,
        "pro": 25.0,
        "enterprise": 60.0,
    }

    def __init__(self, plan, seats):
        self.plan = plan
        self.seats = seats
        self._active = True

    def get_plan(self):
        return self._plan

    def set_plan(self, value):
        if not isinstance(value, str):
            raise TypeError("plan must be a string")

        value = value.strip().lower()

        if value not in self.ALLOWED_PLANS:
            raise ValueError(f"unsupported plan: {value!r}")

        if "_seats" in self.__dict__:
            max_seats = self.MAX_SEATS[value]

            if max_seats is not None and self._seats > max_seats:
                raise ValueError(
                    f"cannot switch to {value}; "
                    f"{self._seats} seats exceeds its limit of {max_seats}"
                )

        self._plan = value

    def get_seats(self):
        return self._seats

    def set_seats(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("seats must be an integer")

        if value <= 0:
            raise ValueError("seats must be positive")

        max_seats = self.MAX_SEATS[self.plan]

        if max_seats is not None and value > max_seats:
            raise ValueError(
                f"{self.plan} plan allows at most {max_seats} seats"
            )

        self._seats = value

    def get_monthly_price(self):
        return self.PRICE_PER_SEAT[self.plan] * self.seats

    def get_active(self):
        return self._active

    def cancel(self):
        if not self._active:
            raise ValueError("subscription is already cancelled")
        self._active = False

    def reactivate(self):
        if self._active:
            raise ValueError("subscription is already active")
        self._active = True

    plan = property(get_plan, set_plan)
    seats = property(get_seats, set_seats)
    monthly_price = property(get_monthly_price)
    active = property(get_active)

In [84]:
sub = Subscription("pro", 12)

print(sub.active)

sub.cancel()
print(sub.active)

sub.reactivate()
print(sub.active)

True
False
True


In [85]:
show_exception(setattr, sub, "active", False)

AttributeError: property 'active' of 'Subscription' object has no setter


The absence of a setter makes the public property read-only.

State changes still happen, but only through the operations that express the business meaning.

## Capstone Step 7 — Introspection

Let us inspect what belongs to the instance and what belongs to the class.

In [86]:
print("INSTANCE DICTIONARY")
print(sub.__dict__)

print("\nCLASS PROPERTY OBJECTS")
print("plan:", Subscription.__dict__["plan"])
print("seats:", Subscription.__dict__["seats"])
print("monthly_price:", Subscription.__dict__["monthly_price"])
print("active:", Subscription.__dict__["active"])

INSTANCE DICTIONARY
{'_plan': 'pro', '_seats': 12, '_active': True}

CLASS PROPERTY OBJECTS
plan: <property object at 0x0000024F7F312930>
seats: <property object at 0x0000024F7F322700>
monthly_price: <property object at 0x0000024F7F320400>
active: <property object at 0x0000024F7F320D60>


In [87]:
print("monthly_price getter:", Subscription.monthly_price.fget)
print("monthly_price setter:", Subscription.monthly_price.fset)

print("active getter:", Subscription.active.fget)
print("active setter:", Subscription.active.fset)

monthly_price getter: <function Subscription.get_monthly_price at 0x0000024F7F315940>
monthly_price setter: None
active getter: <function Subscription.get_active at 0x0000024F7F3159E0>
active setter: None


`monthly_price` and `active` have getters but no setters.

That is why assigning to them raises `AttributeError`.

## Capstone Step 8 — Final behavior checks

In [88]:
basic = Subscription("basic", 5)
pro = Subscription("pro", 100)
enterprise = Subscription("enterprise", 500)

assert basic.monthly_price == 50.0
assert pro.monthly_price == 2500.0
assert enterprise.monthly_price == 30000.0

assert basic.active is True
basic.cancel()
assert basic.active is False

print("All capstone checks passed.")

All capstone checks passed.


# Final Review

Across these problems, we repeatedly saw the same core structure:

```text
public property name
        ↓
getter / setter / deleter logic
        ↓
instance-specific backing state
```

The **property object** is normally stored in the class dictionary.

The **managed value** is normally stored in the instance dictionary under a separate backing
name.

### Questions you should now be able to answer

1. Why can a plain attribute later be replaced with a property without changing dotted syntax?
2. Why is `_name` commonly used as backing storage for a property named `name`?
3. What does a property getter actually do when `obj.name` is evaluated?
4. What does `setattr(obj, "name", value)` do when `name` is a property?
5. Why does a class-level property still exist after `del obj.name`?
6. Why can an instance dictionary key named `"name"` fail to shadow a property?
7. How can two properties expose different views of one canonical stored value?
8. What initialization problems appear when two setters depend on each other?
9. When should a property be read-only?
10. When is a regular method clearer than a setter?

# Extra Problems — Try Before Looking Back at the Solutions

These are intentionally left as practice prompts.

### Extra Problem A — Percentage

Create a `percentage` property that:

- accepts integers and floats;
- rejects `bool`;
- stores a float;
- enforces `0 <= percentage <= 100`.

Then inspect the instance and class dictionaries.

### Extra Problem B — Full Name

Store `_first_name` and `_last_name`.

Add a read-only `full_name` property that joins them.

Then make `first_name` and `last_name` writable properties with non-empty-string validation.

Demonstrate that `full_name` changes automatically without storing `_full_name`.

### Extra Problem C — Revocable API Key

Create an API key property with getter, setter, and deleter.

After deletion:

- `obj.__dict__` should no longer contain the backing value;
- reading the property should raise `AttributeError`;
- assigning a new key should work;
- the property object should still appear in `type(obj).__dict__`.

### Extra Problem D — Cross-Property Constraint

Create `min_value` and `max_value` properties.

Preserve the invariant:

```text
min_value <= max_value
```

Test changing both directions after construction.

### Extra Problem E — Read-Only Computation

Create a circle with a validated `radius` property and read-only properties for:

- `diameter`
- `circumference`
- `area`

Store only the radius.

Verify that the three computed values update when radius changes.

## Closing idea

The important question is not simply:

> "Can I turn this into a property?"

A better question is:

> "Does managed attribute syntax make this object easier to use while preserving its
> invariants?"

Properties are most effective when they keep the public interface simple while moving
validation, normalization, computation, or controlled state management behind that interface.